In [ ]:
%load_ext autoreload
%autoreload 2

from spatial_tcr.utils import setup_plotting, switch_cwd_to_root

switch_cwd_to_root()

figure_dir = "figures/revision/figure-1"
setup_plotting(figure_dir, display_dpi=300, save_dpi=300)

import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc

from spatial_tcr.tcr import get_tcr_genes
from spatial_tcr.utils import (
    aggregate_gene_expression,
)

In [ ]:
data_dir = "data/xenium/processed"
path = f"{data_dir}/06.2-kidney_tcr_filtered.h5ad"
adata = sc.read_h5ad(path)

In [ ]:
tcell_keys = ["CD4+", "TFH", "Tregs", "MAIT", "CD8+", "NK/NKT", "gdT"]

assert all(tcell_key in adata.obs["tcell_subtype"].unique() for tcell_key in tcell_keys)

# define broad and fine cell types
adata.obs["cell_type_l1"] = adata.obs["cell_type_no_tcr"].astype(str)
t_mask = adata.obs["cell_type_l1"] == "T"
adata.obs.loc[t_mask, "cell_type_l1"] = adata.obs.loc[t_mask, "tcell_subtype"]
adata.obs.loc[adata.obs["cell_type_l1"].isin(tcell_keys), "cell_type_l1"] = "T"

adata.obs["cell_type_l2"] = adata.obs["tcell_subtype"].copy()

# merge TFH with CD4+
mapping = {
    "TFH": "CD4+",
    "NK/NKT": "NKT-like",
}
adata.obs["cell_type_l2"] = (
    adata.obs["cell_type_l2"].astype(str).replace(mapping).astype("category")
)

In [ ]:
av_genes, bv_genes, dv_genes, gv_genes, tv_genes = get_tcr_genes(adata)

In [ ]:
av_genes, bv_genes, dv_genes, gv_genes, tv_genes = get_tcr_genes(adata)
trv_mapping = {
    "TRAV": [g for g in av_genes if g != "TRAV1-2"],
    "TRBV": bv_genes,
    "TRDV": dv_genes,
    "TRGV": gv_genes,
}

ad_merged = aggregate_gene_expression(adata, trv_mapping)
ad_merged.layers["counts"] = ad_merged.X.copy()

In [ ]:
sc.pp.normalize_total(ad_merged)
sc.pp.log1p(ad_merged)

In [ ]:
t_mask = ad_merged.obs["cell_type_l1"] == "T"
ad_merged.obs["cell_type_l0"] = ad_merged.obs["cell_type_no_tcr"].astype(str)
ad_merged.obs.loc[~t_mask, "cell_type_l0"] = "other"
ad_merged.obs.loc[t_mask, "cell_type_l0"] = "T cell"

var_names = [
    # "CD3D",
    "CD3E",
    # "CD3G",
    "TRAC",
    "TRBC1",
    "TRBC2",
    "TRAV",
    "TRBV",
    "TRDV",
    "TRGV",
]
fig = sc.pl.dotplot(
    ad_merged, var_names, groupby="cell_type_l0", swap_axes=True, show=False
)

# dp = sc.pl.dotplot(
#     ad_merged,
#     var_names,
#     groupby="cell_type_l0",
#     return_fig=True,
#     # standard_scale="var",
#     cmap="Reds",
#     swap_axes=True,
# )
# dp.add_totals().style(dot_edge_color="black", dot_edge_lw=0.5).show()
plt.savefig(
    os.path.join(figure_dir, "tcr_distribution.pdf"), dpi=300, bbox_inches="tight"
)

## TCR distribution t cell subtypes

In [ ]:
figure_dir = "figures/revision/figure-2"
setup_plotting(figure_dir, display_dpi=300, save_dpi=300)

In [ ]:
ad_t = ad_merged[ad_merged.obs["cell_type_l1"] == "T"].copy()
ct_order = ["CD4+", "CD8+", "Tregs", "MAIT", "NKT-like", "gdT"]
ad_t.obs["cell_type_l2"] = pd.Categorical(
    ad_t.obs["cell_type_l2"], categories=ct_order, ordered=True
)


var_names = [
    # "CD3D",
    # "CD3E",
    # "CD3G",
    "TRAC",
    "TRBC1",
    "TRBC2",
    "TRAV",
    "TRBV",
    "TRDV",
    "TRGV",
]
fig = sc.pl.dotplot(
    ad_t,
    var_names,
    groupby="cell_type_l2",
    swap_axes=True,
    show=False,
    standard_scale="var",
    dot_min=0.1,
    dot_max=1.0,
)

# dp = sc.pl.dotplot(
#     ad_merged,
#     var_names,
#     groupby="cell_type_l0",
#     return_fig=True,
#     # standard_scale="var",
#     cmap="Reds",
#     swap_axes=True,
# )
# dp.add_totals().style(dot_edge_color="black", dot_edge_lw=0.5).show()
save_path = os.path.join(figure_dir, "tcr_distribution_tcell_subtypes.pdf")
print(f"Saving to {save_path}")
plt.savefig(
    save_path,
    dpi=300,
    bbox_inches="tight",
)

In [ ]:
ad_merged.obs["t_vs_rest"] = ad_merged.obs["cell_type_l2"].astype(str)
ad_merged.obs.loc[~(ad_merged.obs["cell_type_l1"] == "T"), "t_vs_rest"] = "other"
ct_order = ["CD4+", "CD8+", "Tregs", "MAIT", "NKT-like", "gdT", "other"]
ad_merged.obs["t_vs_rest"] = pd.Categorical(
    ad_merged.obs["t_vs_rest"], categories=ct_order, ordered=True
)

In [ ]:
var_names = [
    # "CD3D",
    # "CD3E",
    # "CD3G",
    "TRAC",
    "TRBC1",
    "TRBC2",
    "TRAV",
    "TRBV",
    "TRDV",
    "TRGV",
]

var_names = {
    "canonical marker genes": [
        "CD3E",
        "CD4",
        "CD8A",
        "TIGIT",
        "FOXP3",
        "KLRB1",
        "NKG7",
        "GNLY",
        "TRAV1-2",
    ],
    "TCR genes": [
        "TRDC",
        # "TRGC1",
        "TRGC2",
        "TRAC",
        # "TRBC1",
        "TRBC2",
        "TRAV",
        "TRBV",
        "TRDV",
        "TRGV",
    ],
}

In [ ]:
ad_merged.var_names

In [ ]:
plt.rcParams["figure.figsize"] = (5, 2.5)


dp = sc.pl.dotplot(
    # ad_t,
    ad_merged,
    var_names,
    # groupby="cell_type_l2",
    groupby="t_vs_rest",
    swap_axes=False,
    show=False,
    standard_scale="var",
    dot_min=0.0,
    dot_max=1.0,
    var_group_rotation=0,
    return_fig=True,
    figsize=(6, 1.8),
)
# dp.add_totals().style(dot_edge_color="black", dot_edge_lw=0.5)
dp.make_figure()
axg = dp.get_axes()["gene_group_ax"]

# bracket geometry in axis-fraction y, data x
y_top = 0.6
y_bottom = 0.15  # where the vertical lines stop (tweak)
lw = 1.0

x = 0
for i, (_, genes) in enumerate(var_names.items()):
    x_start = x + i * 0.25
    x_end = x + len(genes) + (i - 1) * 0.25

    tr = axg.get_xaxis_transform()  # x in data, y in axes fraction

    # top horizontal
    axg.plot(
        [x_start, x_end],
        [y_top, y_top],
        transform=tr,
        color="black",
        lw=lw,
        clip_on=False,
    )
    # left vertical
    axg.plot(
        [x_start, x_start],
        [y_bottom, y_top],
        transform=tr,
        color="black",
        lw=lw,
        clip_on=False,
    )
    # right vertical
    axg.plot(
        [x_end, x_end],
        [y_bottom, y_top],
        transform=tr,
        color="black",
        lw=lw,
        clip_on=False,
    )

    x += len(genes)
for t in axg.texts:
    t.set_fontsize(8)  # pick your size

ax_main = dp.get_axes()["mainplot_ax"]
ax_main.tick_params(axis="x", labelsize=6)
ax_main.tick_params(axis="y", labelsize=6)

axes = dp.get_axes()

# sizes legend
ax_size = axes["size_legend_ax"]
for t in ax_size.texts:
    t.set_fontsize(6)

# color legend
ax_col = axes["color_legend_ax"]
for t in ax_col.texts:
    t.set_fontsize(6)

for ax in (ax_size, ax_col):
    ax.tick_params(labelsize=6)
    ax.set_title(ax.get_title(), fontsize=6)

plt.savefig(
    os.path.join(figure_dir, "tcr_distribution+markers_tcell_subtypes.pdf"),
    dpi=300,
    bbox_inches="tight",
)

In [ ]:
axg.set_title("TCR distribution in kidney samples", fontsize=10)

## Plot as network

In [ ]:
def av_bv_cooccurrence(adata):
    av_genes, bv_genes = get_tcr_genes(adata)[0:2]
    var_names = av_genes + bv_genes
    X = adata[:, var_names].X.toarray().astype(bool).astype(int)
    counts = X.T @ X
    counts_df = pd.DataFrame(counts, index=var_names, columns=var_names)
    n_cells_per_gene = pd.Series(X.sum(axis=0), index=var_names)
    return counts_df, n_cells_per_gene

In [ ]:
samples = adata.obs["sample"].unique().tolist()
av_genes, bv_genes = get_tcr_genes(adata)[0:2]

for sample in samples:
    ad_sub = adata[adata.obs["sample"] == sample]
    counts_df, n_cells_per_gene = av_bv_cooccurrence(ad_sub)
    break
counts_df

In [ ]:
counts_df, n_cells_per_gene = av_bv_cooccurrence(adata)

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx

# ---- build the bipartite graph -------------------------------------------
G = nx.Graph()
G.add_nodes_from(av_genes, bipartite="av")
G.add_nodes_from(bv_genes, bipartite="bv")

for av in av_genes:
    for bv in bv_genes:  # only AV-to-BV links
        w = counts_df.loc[av, bv]
        if w:  # skip zero co-occurrences
            G.add_edge(av, bv, weight=w)

# ---- place all nodes on a single circle ----------------------------------
ordered = av_genes + bv_genes
angles = np.linspace(0, 2 * np.pi, len(ordered), endpoint=False)
pos = {n: (np.cos(a), np.sin(a)) for n, a in zip(ordered, angles)}

# ---- draw ----------------------------------------------------------------
edge_w = np.array([d["weight"] for *_, d in G.edges(data=True)])
widths = 0.0 + 4.5 * edge_w / edge_w.max()  # 0.5–5 pt scaling

fig, ax = plt.subplots(figsize=(10, 10))
# Color nodes by type
node_colors = ["lightblue" if n in av_genes else "lightcoral" for n in G.nodes()]
# Scale node sizes based on number of cells expressing each gene
node_sizes = [1000 * n_cells_per_gene[n] / n_cells_per_gene.max() for n in G.nodes()]

nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color=node_colors, ax=ax)
nx.draw_networkx_edges(G, pos, width=widths, alpha=0.8, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=5, ax=ax, font_weight="bold")

# plt.title(f"TCR distribution in {sample}")
plt.axis("off")
plt.tight_layout()
plt.show()